# Multi-Bureau Credit Pipeline

This notebook demonstrates a pipeline that scores applicants against two separate credit bureaus, joins the results, and makes a combined final decision.

Key concepts: `FrameRef` for routing named frames, `JoinModule` with module operands, config serialisation.


In [ ]:
%pip install -q decider


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import polars as pl


In [ ]:
from decider.modules.functional import generate_from_functions
from decider.modules._ext import register_graph_module, GraphModule
from decider.modules.primitives.join import JoinModule, FrameRef
from decider.modules.primitives.sequential import SequentialModule


## Sample data

Each bureau provides its own DataFrame for the same applicants.


In [ ]:
bureau1_data = pl.DataFrame({
    "customer_id":  ["C001", "C002", "C003", "C004"],
    "b1_enquiries": [2,       8,      1,       5],
})

bureau2_data = pl.DataFrame({
    "customer_id": ["C001", "C002", "C003", "C004"],
    "b2_balance":  [5_000,  18_000,  1_200,  9_500],
    "b2_limit":    [10_000, 20_000,  8_000, 12_000],
})

print("Bureau 1:"); print(bureau1_data)
print("\nBureau 2:"); print(bureau2_data)


## Bureau scorers

Each bureau has its own scoring module that takes the bureau's raw columns.


In [ ]:
def b1_score(b1_enquiries: pl.Expr) -> pl.Expr:
    return b1_enquiries * -2 + 700

def b1_default_flag(b1_score: pl.Expr) -> pl.Expr:
    return b1_score < 600

Bureau1Scorer = generate_from_functions("bureau1_scorer", b1_score, b1_default_flag)
register_graph_module(Bureau1Scorer)
bureau1_scorer = Bureau1Scorer(name="bureau1_scorer")

print(bureau1_scorer({"input": bureau1_data}))


In [ ]:
def b2_score(b2_balance: pl.Expr, b2_limit: pl.Expr) -> pl.Expr:
    return (b2_balance / b2_limit) * -300 + 700

def b2_default_flag(b2_score: pl.Expr) -> pl.Expr:
    return b2_score < 600

Bureau2Scorer = generate_from_functions("bureau2_scorer", b2_score, b2_default_flag)
register_graph_module(Bureau2Scorer)
bureau2_scorer = Bureau2Scorer(name="bureau2_scorer")

print(bureau2_scorer({"input": bureau2_data}))


## Join bureau outputs

`FrameRef` routes each named input frame to its scorer. `JoinModule` then joins the two resulting frames on `customer_id`.


In [ ]:
# Wrap each scorer so it reads from its own named input frame
b1_pipeline = FrameRef(name="bureau1") | bureau1_scorer
b2_pipeline = FrameRef(name="bureau2") | bureau2_scorer

bureau_join = JoinModule(
    name="bureau_join",
    left=b1_pipeline,
    right=b2_pipeline,
    on="customer_id",
    how="left",
)

joined = bureau_join({"bureau1": bureau1_data, "bureau2": bureau2_data})
print(joined)


## Combined decision

Average the two bureau scores and apply a threshold for the final accept/decline.


In [ ]:
def combined_score(b1_score: pl.Expr, b2_score: pl.Expr) -> pl.Expr:
    return ((b1_score + b2_score) / 2).round(2)

def final_decision(combined_score: pl.Expr) -> pl.Expr:
    return combined_score >= 600

CombinedScorer = generate_from_functions("combined_scorer", combined_score, final_decision)
register_graph_module(CombinedScorer)
combined_scorer = CombinedScorer(name="combined_scorer")


In [ ]:
pipeline = bureau_join | combined_scorer

result = pipeline({"bureau1": bureau1_data, "bureau2": bureau2_data})
print(result.select(["customer_id", "b1_score", "b2_score", "combined_score", "final_decision"]))


## Config serialisation & roundtrip

The full pipeline — including `FrameRef`, nested `JoinModule`, and all scorers — serialises to JSON and can be restored exactly.


In [ ]:
import json

cfg = pipeline.model_dump()
print(json.dumps(cfg, indent=2))


In [ ]:
restored_pipeline = GraphModule.model_validate(cfg).root

restored_result = restored_pipeline({"bureau1": bureau1_data, "bureau2": bureau2_data})
print("Outputs match after roundtrip:", result.equals(restored_result))
print(restored_result.select(["customer_id", "combined_score", "final_decision"]))
